# PTB-XL ECG Classification
**Efficient 12-Lead ECG Diagnosis for Resource-Constrained Deployment**

Two models trained and compared:
- `ResNet1D_Full` — strong baseline (~2M params)
- `ResNet1D_Lite` — 8× smaller, edge-deployable

Dataset files: `train_signal.csv`, `train_meta.csv` (+ valid/test equivalents).  
Accelerator: **GPU T4 x2** — set via Notebook Settings → Accelerator → GPU.

In [2]:
import os, warnings, json, time, ast
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import MultiLabelBinarizer
warnings.filterwarnings("ignore")

DATA_DIR   = "/kaggle/input/datasets/khyeh0719/ptb-xl-dataset-reformatted"
BATCH      = 64
EPOCHS     = 40
LR         = 1e-3
DROPOUT    = 0.3
SEED       = 42
SUPERCLASS = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
N_LEADS    = 12
T          = 1000          # 10 s × 100 Hz
device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED); np.random.seed(SEED)

print("device:", device)
print("files:", sorted(os.listdir(DATA_DIR)))

device: cuda
files: ['test_meta.csv', 'test_signal.csv', 'train_meta.csv', 'train_signal.csv', 'valid_meta.csv', 'valid_signal.csv']


## 1. Explore file structure

In [3]:
# ── peek at meta ────────────────────────────────────────────────────────────
meta_tr = pd.read_csv(f"{DATA_DIR}/train_meta.csv")
print("meta columns:", meta_tr.columns.tolist())
print("meta shape  :", meta_tr.shape)
meta_tr.head(3)

meta columns: ['ecg_id', 'age', 'sex', 'height', 'weight', 'nurse', 'site', 'device', 'NORM', 'MI', 'STTC', 'HYP', 'CD', 'sub_NORM', 'sub_IMI', 'sub_STTC', 'sub_NST_', 'sub_LVH', 'sub_LAFB/LPFB', 'sub_RVH', 'sub_RAO/RAE', 'sub_IRBBB', 'sub_IVCD', 'sub_LMI', 'sub_AMI', 'sub__AVB', 'sub_ISCA', 'sub_ISC_', 'sub_SEHYP', 'sub_ISCI', 'sub_CRBBB', 'sub_CLBBB', 'sub_LAO/LAE', 'sub_ILBBB', 'sub_WPW', 'sub_PMI', 'strat_fold']
meta shape  : (17441, 37)


,ecg_id,age,sex,height,weight,nurse,site,device,NORM,MI,...,sub_ISC_,sub_SEHYP,sub_ISCI,sub_CRBBB,sub_CLBBB,sub_LAO/LAE,sub_ILBBB,sub_WPW,sub_PMI,strat_fold
0,1,56.0,1,NaN,63.0,2.0,0.0,CS-12 E,1,0,...,0,0,0,0,0,0,0,0,0,3
1,2,19.0,0,NaN,70.0,2.0,0.0,CS-12 E,1,0,...,0,0,0,0,0,0,0,0,0,2
2,3,37.0,1,NaN,69.0,2.0,0.0,CS-12 E,1,0,...,0,0,0,0,0,0,0,0,0,5


In [4]:
# ── peek at signal ──────────────────────────────────────────────────────────
sig_peek = pd.read_csv(f"{DATA_DIR}/train_signal.csv", nrows=2)
print("signal shape (2 rows):", sig_peek.shape)
print("first 6 col names    :", sig_peek.columns.tolist()[:6])
print("last  6 col names    :", sig_peek.columns.tolist()[-6:])

signal shape (2 rows): (2, 13)
first 6 col names    : ['ecg_id', 'channel-0', 'channel-1', 'channel-2', 'channel-3', 'channel-4']
last  6 col names    : ['channel-6', 'channel-7', 'channel-8', 'channel-9', 'channel-10', 'channel-11']


## 2. Load signals + labels

In [17]:
import gc

def load_signals(split):
    path = f"{DATA_DIR}/{split}_signal.csv"
    lead_cols = [f"channel-{i}" for i in range(N_LEADS)]
    print(f"  loading {split}_signal.csv …", flush=True)
    t0 = time.time()
    try:
        df = pd.read_csv(path,
                         usecols=lead_cols,          # skip ecg_id entirely
                         dtype=np.float32)           # read as float32 directly
        arr = df.values                              # (N*T, 12) already float32
        del df; gc.collect()                         # free DataFrame immediately
        n_ecgs = len(arr) // T
        arr = arr[:n_ecgs * T].reshape(n_ecgs, T, N_LEADS).transpose(0, 2, 1)
        print(f"  → {arr.shape}  ({time.time()-t0:.1f}s)", flush=True)
        return arr
    except Exception as e:
        print(f"  ERROR: {type(e).__name__}: {e}", flush=True)
        raise

print("Loading signals:")
X_tr = load_signals("train")
X_vl = load_signals("valid")
X_te = load_signals("test")
print("Done.")

Loading signals:
  loading train_signal.csv …
  → (17441, 12, 1000)  (22.2s)
  loading valid_signal.csv …
  → (2193, 12, 1000)  (2.9s)
  loading test_signal.csv …
  → (2203, 12, 1000)  (2.9s)
Done.


In [14]:
# ── label extraction ────────────────────────────────────────────────────────
def extract_labels(meta):
    # try common column names for superclass labels
    for col in ['diagnostic_superclass', 'superdiagnostic',
                'superclass', 'label', 'diagnosis']:
        if col in meta.columns:
            print(f"  using label column: '{col}'")
            def parse(v):
                if isinstance(v, (list, np.ndarray)): return list(v)
                try:    return ast.literal_eval(str(v))
                except: return [x.strip() for x in str(v).split(',') if x.strip()]
            labels = meta[col].apply(parse).tolist()
            mlb = MultiLabelBinarizer(classes=SUPERCLASS)
            return mlb.fit_transform(labels).astype(np.float32)

    # if superclass columns already one-hot encoded
    present = [c for c in SUPERCLASS if c in meta.columns]
    if present:
        print(f"  using one-hot columns: {present}")
        Y = np.zeros((len(meta), len(SUPERCLASS)), dtype=np.float32)
        for i, c in enumerate(SUPERCLASS):
            if c in meta.columns:
                Y[:, i] = meta[c].fillna(0).values.astype(np.float32)
        return Y

    # last resort: print columns and raise
    raise ValueError(
        f"Cannot find label column. Available: {meta.columns.tolist()}")

print("Extracting labels:")
meta_vl = pd.read_csv(f"{DATA_DIR}/valid_meta.csv")
meta_te = pd.read_csv(f"{DATA_DIR}/test_meta.csv")

Y_tr = extract_labels(meta_tr)
Y_vl = extract_labels(meta_vl)
Y_te = extract_labels(meta_te)

print(f"Y_tr {Y_tr.shape}  pos rates: {np.round(Y_tr.mean(0),3)}")
print(f"Y_vl {Y_vl.shape}  Y_te {Y_te.shape}")
for s, r in zip(SUPERCLASS, Y_tr.mean(0)):
    print(f"  {s}: {r*100:.1f}%")

Extracting labels:
  using one-hot columns: ['NORM', 'MI', 'STTC', 'CD', 'HYP']
  using one-hot columns: ['NORM', 'MI', 'STTC', 'CD', 'HYP']
  using one-hot columns: ['NORM', 'MI', 'STTC', 'CD', 'HYP']
Y_tr (17441, 5)  pos rates: [0.436 0.252 0.24  0.224 0.122]
Y_vl (2193, 5)  Y_te (2203, 5)
  NORM: 43.6%
  MI: 25.2%
  STTC: 24.0%
  CD: 22.4%
  HYP: 12.2%


In [18]:
print(X_tr.shape, X_vl.shape, X_te.shape)

(17441, 12, 1000) (2193, 12, 1000) (2203, 12, 1000)


## 3. Dataset + normalisation

In [19]:
# z-score normalise per lead using training statistics only
mu = X_tr.mean(axis=(0, 2), keepdims=True)   # (1, 12, 1)
sd = X_tr.std(axis=(0, 2),  keepdims=True) + 1e-6
X_tr = (X_tr - mu) / sd
X_vl = (X_vl - mu) / sd
X_te = (X_te - mu) / sd
print(f"After normalisation — X_tr mean: {X_tr.mean():.4f}  std: {X_tr.std():.4f}")

class ECGDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.tensor(X)
        self.Y = torch.tensor(Y)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.Y[i]

tr_dl = DataLoader(ECGDataset(X_tr, Y_tr), BATCH, shuffle=True,  num_workers=2, pin_memory=True)
vl_dl = DataLoader(ECGDataset(X_vl, Y_vl), BATCH, shuffle=False, num_workers=2, pin_memory=True)
te_dl = DataLoader(ECGDataset(X_te, Y_te), BATCH, shuffle=False, num_workers=2, pin_memory=True)
print(f"Batches — train:{len(tr_dl)}  val:{len(vl_dl)}  test:{len(te_dl)}")

After normalisation — X_tr mean: -0.0000  std: 1.0000
Batches — train:273  val:35  test:35


## 4. Model architectures

In [20]:
class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=7, stride=1, drop=0.2):
        super().__init__()
        pad = kernel // 2
        self.net = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel, stride=stride, padding=pad, bias=False),
            nn.BatchNorm1d(out_ch), nn.ReLU(),
            nn.Dropout(drop),
            nn.Conv1d(out_ch, out_ch, kernel, padding=pad, bias=False),
            nn.BatchNorm1d(out_ch),
        )
        self.skip = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, 1, stride=stride, bias=False),
            nn.BatchNorm1d(out_ch)
        ) if (in_ch != out_ch or stride != 1) else nn.Identity()
        self.act = nn.ReLU()

    def forward(self, x):
        return self.act(self.net(x) + self.skip(x))


class ResNet1D(nn.Module):
    def __init__(self, n_leads=12, n_classes=5, channels=None, drop=DROPOUT):
        super().__init__()
        ch = channels
        self.stem = nn.Sequential(
            nn.Conv1d(n_leads, ch[0], 15, padding=7, bias=False),
            nn.BatchNorm1d(ch[0]), nn.ReLU(),
        )
        self.blocks = nn.Sequential(
            ResBlock(ch[0], ch[1], stride=2, drop=drop),
            ResBlock(ch[1], ch[1], drop=drop),
            ResBlock(ch[1], ch[2], stride=2, drop=drop),
            ResBlock(ch[2], ch[2], drop=drop),
            ResBlock(ch[2], ch[3], stride=2, drop=drop),
            ResBlock(ch[3], ch[3], drop=drop),
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool1d(1), nn.Flatten(),
            nn.Dropout(drop), nn.Linear(ch[3], n_classes)
        )

    def forward(self, x):
        return self.head(self.blocks(self.stem(x)))


def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

full_model = ResNet1D(channels=[64, 128, 256, 512]).to(device)
lite_model = ResNet1D(channels=[32,  64, 128, 128]).to(device)

print(f"ResNet1D_Full  params : {count_params(full_model):>10,}")
print(f"ResNet1D_Lite  params : {count_params(lite_model):>10,}")
print(f"Size ratio            : {count_params(full_model)/count_params(lite_model):.1f}×")

ResNet1D_Full  params :  8,624,773
ResNet1D_Lite  params :    996,805
Size ratio            : 8.7×


## 5. Training

In [21]:
def train_model(model, name, epochs=EPOCHS, lr=LR):
    opt    = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched  = torch.optim.lr_scheduler.OneCycleLR(
                 opt, max_lr=lr, steps_per_epoch=len(tr_dl), epochs=epochs)
    scaler = GradScaler()
    pos_w  = torch.tensor(
                 (Y_tr == 0).sum(0) / np.maximum((Y_tr == 1).sum(0), 1),
                 dtype=torch.float32).to(device)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    best_auc, best_state, history = 0.0, None, []

    for ep in range(1, epochs + 1):
        model.train(); tr_loss = 0
        for xb, yb in tr_dl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            with autocast():
                loss = loss_fn(model(xb), yb)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            tr_loss += loss.item()

        model.eval(); preds, trues = [], []
        with torch.no_grad():
            for xb, yb in vl_dl:
                preds.append(torch.sigmoid(model(xb.to(device))).cpu().numpy())
                trues.append(yb.numpy())
        preds = np.concatenate(preds); trues = np.concatenate(trues)
        per_auc = [roc_auc_score(trues[:,i], preds[:,i])
                   if trues[:,i].sum() > 0 else float('nan') for i in range(5)]
        macro = np.nanmean(per_auc)
        history.append({'ep': ep, 'loss': tr_loss/len(tr_dl), 'val_auc': macro})
        if macro > best_auc:
            best_auc = macro
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        if ep % 5 == 0 or ep == 1:
            auc_str = "  ".join(f"{s}:{a:.3f}" for s, a in zip(SUPERCLASS, per_auc))
            print(f"[{name}] ep {ep:02d} | loss {tr_loss/len(tr_dl):.4f} "
                  f"| val AUROC {macro:.4f} | {auc_str}")

    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    print(f"[{name}] ✓ best val macro-AUROC: {best_auc:.4f}")
    return model, history

print("=" * 60)
print("Training ResNet1D_Full …")
print("=" * 60)
full_model, full_hist = train_model(full_model, "Full")

Training ResNet1D_Full …
[Full] ep 01 | loss 0.7114 | val AUROC 0.8812 | NORM:0.930  MI:0.866  STTC:0.916  CD:0.869  HYP:0.825
[Full] ep 05 | loss 0.5350 | val AUROC 0.9103 | NORM:0.936  MI:0.900  STTC:0.929  CD:0.908  HYP:0.878
[Full] ep 10 | loss 0.4957 | val AUROC 0.9200 | NORM:0.941  MI:0.910  STTC:0.924  CD:0.927  HYP:0.898
[Full] ep 15 | loss 0.4391 | val AUROC 0.9255 | NORM:0.946  MI:0.924  STTC:0.921  CD:0.928  HYP:0.908
[Full] ep 20 | loss 0.3746 | val AUROC 0.9240 | NORM:0.946  MI:0.926  STTC:0.928  CD:0.923  HYP:0.896
[Full] ep 25 | loss 0.2817 | val AUROC 0.9251 | NORM:0.945  MI:0.926  STTC:0.924  CD:0.929  HYP:0.900
[Full] ep 30 | loss 0.1582 | val AUROC 0.9153 | NORM:0.936  MI:0.924  STTC:0.905  CD:0.913  HYP:0.898
[Full] ep 35 | loss 0.0804 | val AUROC 0.9107 | NORM:0.932  MI:0.921  STTC:0.899  CD:0.913  HYP:0.889
[Full] ep 40 | loss 0.0588 | val AUROC 0.9107 | NORM:0.932  MI:0.921  STTC:0.902  CD:0.910  HYP:0.887
[Full] ✓ best val macro-AUROC: 0.9288


In [22]:
print("=" * 60)
print("Training ResNet1D_Lite …")
print("=" * 60)
lite_model, lite_hist = train_model(lite_model, "Lite")

Training ResNet1D_Lite …
[Lite] ep 01 | loss 0.8121 | val AUROC 0.8716 | NORM:0.920  MI:0.832  STTC:0.900  CD:0.845  HYP:0.861
[Lite] ep 05 | loss 0.5556 | val AUROC 0.9085 | NORM:0.937  MI:0.891  STTC:0.919  CD:0.910  HYP:0.885
[Lite] ep 10 | loss 0.5119 | val AUROC 0.9190 | NORM:0.943  MI:0.903  STTC:0.924  CD:0.925  HYP:0.900
[Lite] ep 15 | loss 0.4570 | val AUROC 0.9275 | NORM:0.948  MI:0.926  STTC:0.933  CD:0.926  HYP:0.905
[Lite] ep 20 | loss 0.4060 | val AUROC 0.9265 | NORM:0.948  MI:0.924  STTC:0.928  CD:0.921  HYP:0.911
[Lite] ep 25 | loss 0.3447 | val AUROC 0.9264 | NORM:0.947  MI:0.930  STTC:0.924  CD:0.924  HYP:0.907
[Lite] ep 30 | loss 0.2663 | val AUROC 0.9208 | NORM:0.945  MI:0.928  STTC:0.922  CD:0.917  HYP:0.892
[Lite] ep 35 | loss 0.2080 | val AUROC 0.9192 | NORM:0.943  MI:0.929  STTC:0.919  CD:0.914  HYP:0.890
[Lite] ep 40 | loss 0.1861 | val AUROC 0.9182 | NORM:0.943  MI:0.928  STTC:0.918  CD:0.913  HYP:0.890
[Lite] ✓ best val macro-AUROC: 0.9286


## 6. Test-set evaluation

In [23]:
def evaluate(model, name, dl):
    model.eval(); preds, trues = [], []
    t0 = time.time()
    with torch.no_grad():
        for xb, yb in dl:
            preds.append(torch.sigmoid(model(xb.to(device))).cpu().numpy())
            trues.append(yb.numpy())
    inf_ms = (time.time() - t0) * 1000 / len(dl.dataset)
    preds  = np.concatenate(preds)
    trues  = np.concatenate(trues)
    per_auc  = [roc_auc_score(trues[:,i], preds[:,i])
                if trues[:,i].sum() > 0 else float('nan') for i in range(5)]
    per_ap   = [average_precision_score(trues[:,i], preds[:,i])
                if trues[:,i].sum() > 0 else float('nan') for i in range(5)]
    print(f"\n── {name} (test set) {'─'*35}")
    print(f"{'Class':<8} {'AUROC':>7}  {'AUPRC':>7}")
    for s, a, p in zip(SUPERCLASS, per_auc, per_ap):
        print(f"  {s:<6} {a:>7.4f}  {p:>7.4f}")
    print(f"  {'MACRO':<6} {np.nanmean(per_auc):>7.4f}  {np.nanmean(per_ap):>7.4f}")
    print(f"  Inference: {inf_ms:.3f} ms/sample  |  Params: {count_params(model):,}")
    return per_auc, inf_ms

full_auc, full_ms = evaluate(full_model, "ResNet1D_Full", te_dl)
lite_auc, lite_ms = evaluate(lite_model, "ResNet1D_Lite", te_dl)


── ResNet1D_Full (test set) ───────────────────────────────────
Class      AUROC    AUPRC
  NORM    0.9472   0.9264
  MI      0.9213   0.8203
  STTC    0.9347   0.8205
  CD      0.9041   0.7975
  HYP     0.9154   0.6786
  MACRO   0.9246   0.8086
  Inference: 0.768 ms/sample  |  Params: 8,624,773

── ResNet1D_Lite (test set) ───────────────────────────────────
Class      AUROC    AUPRC
  NORM    0.9432   0.9226
  MI      0.9239   0.8246
  STTC    0.9324   0.8255
  CD      0.9184   0.8286
  HYP     0.9086   0.6696
  MACRO   0.9253   0.8142
  Inference: 0.222 ms/sample  |  Params: 996,805


## 7. Efficiency summary — core paper table

In [24]:
rows = []
for s, fa, la in zip(SUPERCLASS, full_auc, lite_auc):
    rows.append({
        'Superclass'    : s,
        'Full AUROC'    : round(fa, 4),
        'Lite AUROC'    : round(la, 4),
        'Retention (%)'  : round(100 * la / fa, 1) if fa and fa > 0 else float('nan')
    })
rows.append({
    'Superclass'    : 'MACRO',
    'Full AUROC'    : round(float(np.nanmean(full_auc)), 4),
    'Lite AUROC'    : round(float(np.nanmean(lite_auc)), 4),
    'Retention (%)'  : round(100 * np.nanmean(lite_auc) / np.nanmean(full_auc), 1)
})
summary = pd.DataFrame(rows)
print(summary.to_string(index=False))
print(f"\nFull params : {count_params(full_model):,}")
print(f"Lite params : {count_params(lite_model):,}")
print(f"Size ratio  : {count_params(full_model)/count_params(lite_model):.1f}×")
print(f"Full inference : {full_ms:.3f} ms/sample")
print(f"Lite inference : {lite_ms:.3f} ms/sample")
print(f"Speed ratio : {full_ms/lite_ms:.1f}×")

Superclass  Full AUROC  Lite AUROC  Retention (%)
      NORM      0.9472      0.9432           99.6
        MI      0.9213      0.9239          100.3
      STTC      0.9347      0.9324           99.8
        CD      0.9041      0.9184          101.6
       HYP      0.9154      0.9086           99.3
     MACRO      0.9246      0.9253          100.1

Full params : 8,624,773
Lite params : 996,805
Size ratio  : 8.7×
Full inference : 0.768 ms/sample
Lite inference : 0.222 ms/sample
Speed ratio : 3.5×


## 8. Save outputs

In [25]:
torch.save(full_model.state_dict(), "resnet1d_full.pt")
torch.save(lite_model.state_dict(), "resnet1d_lite.pt")
json.dump({'full': full_hist, 'lite': lite_hist},
          open("training_history.json", "w"), indent=2)
summary.to_csv("results_summary.csv", index=False)
print("Saved: resnet1d_full.pt | resnet1d_lite.pt | training_history.json | results_summary.csv")

Saved: resnet1d_full.pt | resnet1d_lite.pt | training_history.json | results_summary.csv


## 9. Paper notes
**Benchmark:** Wagner et al. 2020 PTB-XL paper reports macro AUROC ~0.93 for ResNet — compare your Full model there.  
**Contribution:** ResNet1D_Lite achieves ≥95% AUROC retention at 8× fewer parameters and faster inference — publishable as efficient ECG classification for low-resource/African clinical settings.  
**Next experiments for the paper:**  
1. Per-lead ablation — can you get acceptable AUROC with 6 leads instead of 12? (reduces hardware cost further)  
2. CPU inference time comparison (relevant for deployment)  
3. Confidence calibration (reliability diagram)